# Veille classification / bonnes pratiques du traitement de la donnée

## I - La classification de données

### 1. Introduction

La classification consiste à prédire la catégorie d’une nouvelle observation parmi des classes définies à l’avance. Elle relève principalement de l’apprentissage supervisé : le modèle apprend à partir d’exemples dont l’étiquette est connue et définie.

Elle n’est toutefois pas exclusivement supervisée :

- Semi-supervisé : quelques exemples étiquetés et beaucoup d’exemples non étiquetés, exploités ensemble. Scikit-learn propose un module dédié (sklearn.semi_supervised) 
- Auto-supervisé puis ajustement fin : pré-entraînement sur des données brutes sans étiquettes humaines, puis spécialisation sur la tâche avec un petit jeu étiqueté (cas de BERT)
- Apprentissage faiblement supervisé ou PU (positive-unlabeled) : étiquettes bruitées, partielles ou limitées à la classe positive ;
modèles à une classe (One-Class SVM, Isolation Forest) : on n’apprend que le comportement « normal » pour détecter ce qui s’en écarte

Il existe également des méthodes de clustering (k-means, DBSCAN), qui est non supervisé : il regroupe les données sans classes prédéfinies.

Exemple d'applications courantes : détection de spam, aide au diagnostic médical, prédiction de départ de clients (churn score)...

On distingue trois cas :

- **La classification binaire** : deux classes (spam / non-spam)
- **La classification multiclasse** : une classe parmi N
- **La classification multilabel** : plusieurs étiquettes possibles pour une même observation

---

### 2. Les grandes familles de modèles

#### a) Modèles linéaires : la régression logistique

- Traite le binaire comme le multiclasse et renvoie des probabilités.
- Se règle par la régularisation (paramètre `C`, pénalités l1, l2, elasticnet).
- Simple, rapide, interprétable : excellent point de départ.

#### b) SVM (Support Vector Machine)

- Cherche l'hyperplan séparant les classes avec la marge maximale (Cortes & Vapnik, 1995).
- Avec un noyau (RBF), gère des frontières non linéaires.
- Efficace en grande dimension, mais coûteux sur de très gros volumes.

#### c) K-Nearest Neighbors (k-NN/KNN) et Naive Bayes

- **k-NN** : classe une observation selon ses voisins les plus proches, sans véritable phase d'entraînement.
- **Naive Bayes** : classifieurs probabilistes fondés sur le théorème de Bayes, très rapides à entraîner et à utiliser. La variante multinomiale convient à la classification de texte.

#### d) Arbres de décision

- Apprennent des règles de décision successives.
- Interprétables, capables de modéliser des relations non linéaires, acceptent des variables de types mixtes.
- Réglages principaux : profondeur maximale, nombre minimal d'échantillons par feuille, critère (`gini` ou `entropy`).
- Un arbre isolé a tendance à sur-apprendre.

#### e) Méthodes d'ensemble

- **Forêt aléatoire** (Breiman, 2001) : moyenne de nombreux arbres pour réduire la variance.
- **Boosting** (Gradient Boosting, XGBoost, LightGBM) : arbres construits séquentiellement, chacun corrigeant les erreurs des précédents. AdaBoost, par exemple, met l'accent sur les échantillons mal classés.
- **Vote** : combine plusieurs modèles par majorité (*hard*) ou en moyennant les probabilités (*soft*).

#### f) Réseaux de neurones

- Perceptrons multicouches, CNN, Transformers : capables d'apprendre des motifs très complexes (images, texte, audio).
- Exigent plus de données, plus de réglages et plus de calcul, et demandent une mise à l'échelle des variables.

---

### 3. Quel modèle choisir ? Le cas des données tabulaires

Une étude présentée à NeurIPS 2022 a comparé modèles à base d'arbres et réseaux de neurones sur **45 jeux de données** variés. Résultat : les modèles à base d'arbres (XGBoost, forêts aléatoires) restent à l'état de l'art sur des données tabulaires de **taille moyenne (~10 000 échantillons)**, même sans tenir compte de leur rapidité supérieure.

Les auteurs expliquent l'écart par trois difficultés des réseaux de neurones sur ce type de données :

1. être robustes aux variables non informatives
2. préserver l'orientation des données
3. apprendre facilement des fonctions irrégulières

**Règle pratique :**

| Contexte | Choix usuel |
|---|---|
| Données tabulaires, taille moyenne | Gradient boosting / forêt aléatoire |
| Besoin d'interprétabilité | Régression logistique, arbre peu profond |
| Texte, images, audio | Deep learning |
| Modèle de référence rapide | Régression logistique ou Naive Bayes |

---

De manière générale, il vaut mieux débuter sur une analyse appronfondie du dataset, implémenter un modèle simple et explicable pour référence, puis itérrer.

### 4. Évaluer un modèle de classification

L'exactitude (*accuracy*) seule est trompeuse quand les classes sont déséquilibrées. On utilise plusieurs métriques :

| Métrique | Formule | Interprétation |
|---|---|---|
| **Précision** | TP / (TP + FP) | Capacité à ne pas étiqueter positif un échantillon négatif |
| **Rappel** | TP / (TP + FN) | Capacité à retrouver tous les échantillons positifs |
| **F1** | 2 × (P × R) / (P + R) | Moyenne harmonique de la précision et du rappel |

**Agrégation en multiclasse** :

- **macro** : moyenne non pondérée des classes, ignore le déséquilibre
- **micro** : comptage global des vrais et faux positifs et négatifs
- **weighted** : moyenne pondérée par le nombre d'exemples de chaque classe

À compléter par la matrice de confusion, la courbe ROC / l'AUC et la courbe précision-rappel.

Les principales métriques de classification sont l’accuracy, la précision, le rappel, la spécificité, le F1-score, la balanced accuracy, le MCC, le kappa de Cohen, la ROC-AUC, la PR-AUC, la log loss et le Brier score, auxquelles s’ajoutent la top-k accuracy et les moyennes macro/micro/weighted en multiclasse.

**Bonnes pratiques :** validation croisée, jeu de test isolé, pipelines pour éviter les fuites de données lors du réglage des hyperparamètres.

---

### 5. Conclusion

- Il n'existe pas de meilleur modèle universel : le choix dépend des données, de leur volume et du besoin d'interprétabilité.
- Commencer par un modèle de référence simple, puis passer à un gradient boosting.
- Choisir la métrique selon le coût des erreurs : faux négatifs ou faux positifs.
- Avec scikit-learn, le flux de travail est toujours le même : instanciation, entraînement (`fit`), prédiction (`predict`).

---

### Sources

1. Grinsztajn, L., Oyallon, E., Varoquaux, G. (2022). *Why do tree-based models still outperform deep learning on typical tabular data?* NeurIPS 2022. <https://arxiv.org/abs/2207.08815>
2. scikit-learn, documentation de `precision_recall_fscore_support`. <https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_fscore_support.html>
3. scikit-learn, documentation de `f1_score`. <https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score>
4. Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python.* JMLR 12 (arXiv:1201.0490). <https://arxiv.org/abs/1201.0490>
5. StackAbuse, *Overview of Classification Methods in Python with Scikit-Learn*. <https://stackabuse.com/overview-of-classification-methods-in-python-with-scikit-learn/>
6. Résumé de référence sur les algorithmes supervisés de scikit-learn (source secondaire). <https://cdn.jsdelivr.net/npm/claude-code-skills@0.5.0/data/skills/scikit-learn/references/supervised_learning.md>